# Revised Chakraborty–Stokes replication: data audit, preprocessing, and Haiyan anomaly detection

This notebook re-evaluates whether the adaptive nighttime-lights workflow of Chakraborty and Stokes (2023) can be transferred to Super Typhoon Haiyan in Samar–Leyte.

The workflow is deliberately sequential:

1. **Inspect the raw inputs before transformation.**
2. **Identify missingness, spatial incompleteness, and radiance contamination.**
3. **Apply transparent preprocessing and visualize the before/after result.**
4. **Test whether each series is sufficiently complete for the published 60-day input and 30-day output design.**
5. **Run a scaled, Haiyan-limited exploratory model only where the sequence design is computationally feasible.**
6. **Suppress recovery interpretation when no negative anomaly is detected, the model is unstable, or observability is inadequate.**

The regional gap-filled series is a methodological benchmark. The GHSL branch uses directly observed `DNB_BRDF_Corrected_NTL` summaries and remains conditional on observability. NTL departures indicate changes in electricity-dependent nocturnal activity; they do not directly measure electricity restoration or community recovery.

Primary reference: [Chakraborty and Stokes (2023)](https://doi.org/10.1016/j.rse.2023.113818), *Remote Sensing of Environment*, 298, 113818. An [open manuscript](https://arxiv.org/abs/2306.08501) is also available.


In [1]:
# ============================================================
# 1. IMPORTS
# ============================================================

from pathlib import Path
import io
import re
import warnings
import zipfile

import numpy as np
import pandas as pd

import plotly.graph_objects as go
from plotly.subplots import make_subplots

from IPython.display import Markdown, display

warnings.filterwarnings("ignore", category=FutureWarning)


In [2]:
# ============================================================
# 2. PATHS, EVENT WINDOWS, AND ANALYTICAL SETTINGS
# ============================================================

PROJECT_DIR = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

DATA_DIR = PROJECT_DIR / "datasets"

GAP_FILLED_CSV = DATA_DIR / "Region VIII_NTL_VNP46A2.csv"
GHSL_DIR = DATA_DIR / "ghsl_masked_ntl_samar_leyte"
GHSL_ZIP = DATA_DIR / "ghsl_masked_ntl_samar_leyte.zip"

# Event and retrospective evaluation windows
EVENT_DATE = pd.Timestamp("2013-11-08")
TRAINING_END = EVENT_DATE - pd.Timedelta(days=90)
ANALYSIS_END = EVENT_DATE + pd.Timedelta(days=365)
HAIYAN_METRIC_END = EVENT_DATE + pd.Timedelta(days=180)
DISPLAY_START = EVENT_DATE - pd.Timedelta(days=365)
DISPLAY_END = ANALYSIS_END

# Published sequence settings
ROLLING_DAYS = 30
INPUT_WINDOW = 60
OUTPUT_WINDOW = 30
TRAIN_FRACTION = 0.80
BATCH_SIZE = 64
MODEL_EPOCHS = {
    "FCNN": 70,
    "CNN": 90,
    "LSTM": 25,
}
ENSEMBLE_WEIGHTS = {
    "FCNN": 0.30,
    "CNN": 0.20,
    "LSTM": 0.50,
}
ANOMALY_TOP_PERCENT = 25
PAPER_MINIMUM_TRAINING_DAYS = 365 * 3
RANDOM_SEED = 42

# Transparent preprocessing rules
GAP_MIN_DAYS_30D = 24
RQ_MIN_OBSERVED_DAYS_30D = 18
VALID_PIXEL_THRESHOLD_PCT = 50.0
RECOVERY_PERSISTENCE_DAYS = 14
DISASTER_DETECTION_DAYS = 90

# Signals
DNB_BAND = "DNB_BRDF_Corrected_NTL"
GAP_FILLED_BAND = "Gap_Filled_DNB_BRDF_Corrected_NTL"

# Plot colours retained from Notebook 2
OBSERVED_COLOR = "#0091FF"
PREDICTED_COLOR = "#000000"
GAP_FILLED_COLOR = "#FF0000"
RQ_RAW_COLOR = "#A8B6CC"
RQ_PROXY_COLOR = "#F28E2B"
RQ_COLOR = "#00C54F"
EVENT_LINE_COLOR = "#0057FF"
ANOMALY_COLOR = "#C62828"
TRAINING_LINE_COLOR = "#7A869A"

SC_COLORSCALE = [
    [0.00, "#F7FCF5"],
    [0.10, "#E5F5E0"],
    [0.25, "#C7E9C0"],
    [0.50, "#74C476"],
    [0.75, "#238B45"],
    [1.00, "#005A32"],
]

if not GAP_FILLED_CSV.exists():
    raise FileNotFoundError(
        f"Gap-filled CSV not found: {GAP_FILLED_CSV}"
    )

if not GHSL_DIR.exists() and not GHSL_ZIP.exists():
    raise FileNotFoundError(
        "Provide either the extracted GHSL directory or its ZIP archive: "
        f"{GHSL_DIR} or {GHSL_ZIP}"
    )

print(f"Gap-filled input: {GAP_FILLED_CSV}")
print(
    "GHSL input: "
    f"{GHSL_DIR if GHSL_DIR.exists() else GHSL_ZIP}"
)
print(f"Training cutoff: {TRAINING_END.date()}")
print(f"Model evaluation ends: {ANALYSIS_END.date()}")


Gap-filled input: /Users/reneprincipejr/Library/CloudStorage/OneDrive-RMITUniversity/02 - CH2 - Disaster Impact and Recovery/blackmarble-disaster-recovery/datasets/Region VIII_NTL_VNP46A2.csv
GHSL input: /Users/reneprincipejr/Library/CloudStorage/OneDrive-RMITUniversity/02 - CH2 - Disaster Impact and Recovery/blackmarble-disaster-recovery/datasets/ghsl_masked_ntl_samar_leyte
Training cutoff: 2013-08-10
Model evaluation ends: 2014-11-08


## 1. Published method and replication constraints

Chakraborty and Stokes use a daily, area-weighted, gap-filled VNP46A2 series and apply a 30-day rolling average. Three neural networks learn to predict a 30-day output from the preceding 60 days. Overlapping forecasts for the same date are reduced by their median, and the FCNN, CNN, and LSTM predictions are combined as a weighted ensemble. Change direction and severity are obtained from the residual $r_t=x_t-\hat{x}_{t,ens}$.

The paper trains on at least three years of stable daily data. That requirement cannot be met for Haiyan because VIIRS begins in January 2012 and Haiyan occurred in November 2013. Any neural-network result here is therefore an **exploratory transfer**, not a complete replication.

The previous notebook also exposed three dataset-specific problems that must be resolved before modelling:

- the regional gap-filled column still contains null dates;
- the GHSL summary tables contain extreme unremoved radiance contamination outside the Haiyan period; and
- a 50% spatial-completeness rule leaves no complete pre-Haiyan 90-day GHSL sequence.

The notebook first demonstrates these limitations. It then restricts modelling to one year after Haiyan, scales each series from its pre-event training distribution, estimates the anomaly threshold from held-out baseline errors, and treats the relaxed GHSL model as diagnostic whenever strict observability fails.


## 2. Load the raw inputs

No smoothing, interpolation, clipping, normalization, or anomaly filtering is applied in this section. The objective is to establish what the supplied files actually contain.


In [3]:
# ============================================================
# 3.1 LOAD RAW REGION VIII BLACK MARBLE
# ============================================================

gap_raw = pd.read_csv(
    GAP_FILLED_CSV,
    parse_dates=["date"],
)

required_gap_columns = {
    "date",
    DNB_BAND,
    GAP_FILLED_BAND,
}

missing_gap_columns = required_gap_columns.difference(
    gap_raw.columns
)

if missing_gap_columns:
    raise KeyError(
        f"Missing gap-filled columns: {sorted(missing_gap_columns)}"
    )

gap_raw = (
    gap_raw
    .sort_values("date")
    .drop_duplicates("date", keep="last")
    .set_index("date")
    .asfreq("D")
)

for column in [DNB_BAND, GAP_FILLED_BAND]:
    gap_raw[column] = pd.to_numeric(
        gap_raw[column],
        errors="coerce",
    )

gap_inventory = pd.DataFrame(
    {
        "Start": [gap_raw.index.min().date()],
        "End": [gap_raw.index.max().date()],
        "Calendar days": [len(gap_raw)],
        "Direct DNB valid days": [gap_raw[DNB_BAND].notna().sum()],
        "Gap-filled valid days": [gap_raw[GAP_FILLED_BAND].notna().sum()],
        "Gap-filled missing days": [gap_raw[GAP_FILLED_BAND].isna().sum()],
        "Negative gap-filled values": [
            gap_raw[GAP_FILLED_BAND].lt(0).sum()
        ],
    }
)

display(gap_inventory)


,Start,End,Calendar days,Direct DNB valid days,Gap-filled valid days,Gap-filled missing days,Negative gap-filled values
0,2012-01-19,2023-06-11,4162,3503,4112,50,0


In [4]:
# ============================================================
# 3.2 LOAD RAW GHSL-MASKED SUMMARY TABLES
# ============================================================

GHSL_MASKS = {
    "10-30": "G1 (codes 10–30)",
    "11-30": "G2 (codes 11–30)",
    "12-30": "G3 (codes 12–30)",
    "13-30": "G4 (codes 13–30)",
    "21-30": "G5 (codes 21–30)",
    "22-30": "G6 (codes 22–30)",
    "23-30": "G7 (codes 23–30)",
    "30": "G8 (code 30)",
}


def ghsl_code_from_name(file_name):
    match = re.search(
        r"DNBBRDF_(.+?)_stats\.csv$",
        Path(file_name).name,
    )

    if match is None:
        raise ValueError(
            f"Cannot identify the GHSL mask from {file_name}"
        )

    return match.group(1)


def load_ghsl_tables():
    tables = {}

    if GHSL_DIR.exists():
        for source in sorted(GHSL_DIR.glob("*.csv")):
            code_value = ghsl_code_from_name(source.name)
            tables[code_value] = pd.read_csv(
                source,
                parse_dates=["date"],
            )

    else:
        with zipfile.ZipFile(GHSL_ZIP) as archive:
            sources = sorted(
                name
                for name in archive.namelist()
                if name.endswith(".csv")
                and not name.startswith("__MACOSX/")
            )

            for source in sources:
                code_value = ghsl_code_from_name(source)

                with archive.open(source) as stream:
                    tables[code_value] = pd.read_csv(
                        io.TextIOWrapper(
                            stream,
                            encoding="utf-8-sig",
                        ),
                        parse_dates=["date"],
                    )

    return tables


ghsl_raw = load_ghsl_tables()

missing_masks = set(GHSL_MASKS).difference(ghsl_raw)

if missing_masks:
    raise KeyError(
        f"Missing GHSL mask tables: {sorted(missing_masks)}"
    )

print(f"Loaded {len(ghsl_raw)} GHSL thematic-mask tables.")


Loaded 8 GHSL thematic-mask tables.


In [5]:
# ============================================================
# 3.3 RAW GHSL INVENTORY AND EXTREME-VALUE AUDIT
# ============================================================

GHSL_NUMERIC_COLUMNS = [
    "NTL_min",
    "NTL_p05",
    "NTL_p25",
    "NTL_median",
    "NTL_p75",
    "NTL_p95",
    "NTL_mean",
    "NTL_max",
    "Mean_by_sum",
    "Valid_px",
]

ghsl_inventory_rows = []
ghsl_extreme_rows = []

for code_value, mask_label in GHSL_MASKS.items():
    frame = (
        ghsl_raw[code_value]
        .sort_values("date")
        .drop_duplicates("date", keep="last")
        .set_index("date")
        .asfreq("D")
    )

    for column in GHSL_NUMERIC_COLUMNS:
        frame[column] = pd.to_numeric(
            frame[column],
            errors="coerce",
        )

    ghsl_raw[code_value] = frame

    valid_mean = frame["NTL_mean"].dropna()
    maximum_date = valid_mean.idxmax()
    maximum_row = frame.loc[maximum_date]

    ghsl_inventory_rows.append(
        {
            "GHSL mask": mask_label,
            "Start": frame.index.min().date(),
            "End": frame.index.max().date(),
            "Observed days (%)": 100 * frame["NTL_mean"].notna().mean(),
            "Median NTL mean": valid_mean.median(),
            "99th percentile NTL mean": valid_mean.quantile(0.99),
            "Maximum NTL mean": valid_mean.max(),
            "Maximum valid pixels": frame["Valid_px"].max(),
        }
    )

    ghsl_extreme_rows.append(
        {
            "GHSL mask": mask_label,
            "Extreme date": maximum_date.date(),
            "NTL mean": maximum_row["NTL_mean"],
            "NTL median": maximum_row["NTL_median"],
            "NTL p95": maximum_row["NTL_p95"],
            "NTL maximum": maximum_row["NTL_max"],
            "Valid pixels": maximum_row["Valid_px"],
            "Mean / median ratio": (
                maximum_row["NTL_mean"]
                / max(maximum_row["NTL_median"], 0.01)
            ),
        }
    )

ghsl_inventory = pd.DataFrame(ghsl_inventory_rows)
ghsl_extreme_audit = pd.DataFrame(ghsl_extreme_rows)

display(
    ghsl_inventory.style.format(
        {
            "Observed days (%)": "{:.1f}",
            "Median NTL mean": "{:.3f}",
            "99th percentile NTL mean": "{:.3f}",
            "Maximum NTL mean": "{:.3f}",
            "Maximum valid pixels": "{:.0f}",
        }
    )
)

display(
    ghsl_extreme_audit.style.format(
        {
            "NTL mean": "{:.3f}",
            "NTL median": "{:.3f}",
            "NTL p95": "{:.3f}",
            "NTL maximum": "{:.3f}",
            "Valid pixels": "{:.0f}",
            "Mean / median ratio": "{:.1f}",
        }
    )
)


,GHSL mask,Start,End,Observed days (%),Median NTL mean,99th percentile NTL mean,Maximum NTL mean,Maximum valid pixels
0,G1 (codes 10–30),2012-01-19,2025-07-21,80.3,0.238,0.941,36.050,88991
1,G2 (codes 11–30),2012-01-19,2025-07-21,80.3,0.238,0.940,36.050,87035
2,G3 (codes 12–30),2012-01-19,2025-07-21,79.9,0.348,1.196,36.050,30549
3,G4 (codes 13–30),2012-01-19,2025-07-21,79.3,0.469,1.497,46.444,15528
4,G5 (codes 21–30),2012-01-19,2025-07-21,78.0,0.563,1.695,63.472,11480
5,G6 (codes 22–30),2012-01-19,2025-07-21,76.2,1.253,3.608,127.018,2782
6,G7 (codes 23–30),2012-01-19,2025-07-21,72.8,2.041,5.488,210.176,1492
7,G8 (code 30),2012-01-19,2025-07-21,62.8,4.150,9.184,337.229,449


,GHSL mask,Extreme date,NTL mean,NTL median,NTL p95,NTL maximum,Valid pixels,Mean / median ratio
0,G1 (codes 10–30),2014-04-15,36.050,36.050,36.050,36.050,1,1.0
1,G2 (codes 11–30),2014-04-15,36.050,36.050,36.050,36.050,1,1.0
2,G3 (codes 12–30),2014-04-15,36.050,36.050,36.050,36.050,1,1.0
3,G4 (codes 13–30),2017-12-02,46.444,0.510,142.604,4589.701,5214,91.0
4,G5 (codes 21–30),2017-12-02,63.472,0.681,271.723,4589.701,3608,93.2
5,G6 (codes 22–30),2017-12-02,127.018,2.123,817.002,3939.214,845,59.8
6,G7 (codes 23–30),2017-12-02,210.176,3.691,1477.930,3939.214,483,56.9
7,G8 (code 30),2017-12-02,337.229,6.029,2086.534,3505.619,264,55.9


In [6]:
# ============================================================
# 4.1 RAW REGION VIII INPUT VISUALIZATION
# ============================================================

fig_gap_raw = go.Figure()

fig_gap_raw.add_trace(
    go.Scatter(
        x=gap_raw.index,
        y=gap_raw[DNB_BAND],
        mode="lines",
        name="Direct DNB-BRDF",
        line=dict(
            color=OBSERVED_COLOR,
            width=1.2,
        ),
        opacity=0.65,
        connectgaps=False,
    )
)

fig_gap_raw.add_trace(
    go.Scatter(
        x=gap_raw.index,
        y=gap_raw[GAP_FILLED_BAND],
        mode="lines",
        name="Gap-filled DNB-BRDF",
        line=dict(
            color=GAP_FILLED_COLOR,
            width=1.6,
        ),
        connectgaps=False,
    )
)

fig_gap_raw.add_vline(
    x=EVENT_DATE.to_pydatetime(),
    line=dict(
        color=EVENT_LINE_COLOR,
        width=2,
        dash="dash",
    ),
)

fig_gap_raw.add_annotation(
    x=(EVENT_DATE + pd.Timedelta(days=5)).to_pydatetime(),
    y=0.05,
    xref="x",
    yref="paper",
    text="Haiyan",
    showarrow=False,
    xanchor="left",
    font=dict(
        color=EVENT_LINE_COLOR,
        size=16,
    ),
)

fig_gap_raw.update_xaxes(
    range=[DISPLAY_START, DISPLAY_END],
    title_text="Date",
)

fig_gap_raw.update_yaxes(
    title_text="Mean radiance (nW cm⁻² sr⁻¹)",
)

fig_gap_raw.update_layout(
    template="plotly_white",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="white",
    width=1200,
    height=560,
    title=dict(
        text="Raw Region VIII direct and gap-filled nighttime lights",
        x=0.5,
        xanchor="center",
        font=dict(size=24),
    ),
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="center",
        x=0.5,
        font=dict(size=15),
    ),
    font=dict(
        family="Arial",
        size=16,
        color="#243B5A",
    ),
    margin=dict(l=100, r=60, t=120, b=70),
    hovermode="x unified",
)

fig_gap_raw.show()

display(
    pd.DataFrame(
        {
            "Missing gap-filled date": (
                gap_raw.index[
                    gap_raw[GAP_FILLED_BAND].isna()
                ]
            )
        }
    )
)


,Missing gap-filled date
0,2012-01-27
1,2012-02-18
2,2012-02-19
3,2012-03-10
4,2012-03-24
5,2012-03-25
6,2012-03-26
7,2012-03-28
8,2012-06-22
9,2012-11-22


**Interpretation.** The band is labelled gap-filled, but the regional CSV still contains null dates. A strict 30-of-30 rolling mean would turn each isolated null into a 30-day gap and would subsequently remove many 60-day model inputs. The missing dates must therefore remain visible and be handled by an explicit rolling-support rule.


In [7]:
# ============================================================
# 4.2 RAW G7 SUMMARY VISUALIZATION
# ============================================================

g7_label = GHSL_MASKS["23-30"]
g7_raw = ghsl_raw["23-30"]

fig_g7_raw = go.Figure()

for column, label, color, width in [
    ("NTL_mean", "Daily NTL mean", OBSERVED_COLOR, 1.4),
    ("NTL_median", "Daily NTL median", RQ_COLOR, 1.4),
    ("NTL_p95", "Daily NTL p95", RQ_PROXY_COLOR, 1.1),
]:
    positive_values = g7_raw[column].where(
        g7_raw[column].gt(0)
    )

    fig_g7_raw.add_trace(
        go.Scatter(
            x=g7_raw.index,
            y=positive_values,
            mode="lines",
            name=label,
            line=dict(
                color=color,
                width=width,
            ),
            connectgaps=False,
        )
    )

fig_g7_raw.add_vline(
    x=EVENT_DATE.to_pydatetime(),
    line=dict(
        color=EVENT_LINE_COLOR,
        width=2,
        dash="dash",
    ),
)

extreme_g7_date = g7_raw["NTL_mean"].idxmax()

fig_g7_raw.add_annotation(
    x=extreme_g7_date,
    y=g7_raw.loc[extreme_g7_date, "NTL_mean"],
    text="Extreme unremoved radiance contamination",
    showarrow=True,
    arrowhead=2,
    ax=-120,
    ay=-60,
    font=dict(
        color=ANOMALY_COLOR,
        size=14,
    ),
)

fig_g7_raw.update_xaxes(title_text="Date")

fig_g7_raw.update_yaxes(
    title_text="Radiance (log scale)",
    type="log",
)

fig_g7_raw.update_layout(
    template="plotly_white",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="white",
    width=1200,
    height=570,
    title=dict(
        text=f"Raw reliability-qualified summary statistics: {g7_label}",
        x=0.5,
        xanchor="center",
        font=dict(size=24),
    ),
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="center",
        x=0.5,
        font=dict(size=15),
    ),
    font=dict(
        family="Arial",
        size=16,
        color="#243B5A",
    ),
    margin=dict(l=100, r=60, t=120, b=70),
    hovermode="x unified",
)

fig_g7_raw.show()


**Interpretation.** The GHSL tables are direct-observation summaries, but their daily means are not consistently protected from extreme spatial contamination. The log-scale diagnostic shows whether the mean, median, and p95 diverge. These extremes cannot be passed directly to an unscaled neural network. They must either be removed at pixel level or excluded from the event-specific modelling period.


## 3. Transparent preprocessing

The following rules are applied and retained as columns:

### Gap-filled benchmark

- non-finite and negative radiances are invalid;
- no daily value is interpolated;
- the 30-day mean is retained when at least 24 of 30 gap-filled dates are available; and
- 30-day availability is plotted as data support, not ground observability.

### Reliability-qualified GHSL summaries

The pixel-level p95 clamp used in the full reliability-qualified workflow cannot be reconstructed exactly from daily summary tables. A traceable p95-winsorized mean approximation is therefore calculated from the supplied spatial quantiles:

$$
\widetilde{\mu}_{p95}
=0.025q_{0}+0.125q_{05}+0.225q_{25}
+0.250q_{50}+0.225q_{75}+0.150q_{95}.
$$

This is the trapezoidal integral of the available quantile function after values above p95 are capped at p95. It is a summary-level approximation, not a replacement for pixel-level clipping.

- a relaxed modelling series retains days with at least one valid pixel and requires 18 observed days per trailing 30-day mean;
- a strict evidential series first requires at least 50% spatial completeness and then the same 18-of-30 temporal rule; and
- the relaxed series may support a diagnostic model, but recovery interpretation still requires the strict observability gate.

The analysis ends one year after Haiyan so that later contamination and long-term concept drift cannot determine the event threshold.


In [8]:
# ============================================================
# 5.1 PREPROCESS THE GAP-FILLED BENCHMARK
# ============================================================

gap_prepared = gap_raw.loc[:ANALYSIS_END].copy()

gap_prepared["ntl_raw"] = gap_prepared[
    GAP_FILLED_BAND
].where(
    np.isfinite(gap_prepared[GAP_FILLED_BAND])
    & gap_prepared[GAP_FILLED_BAND].ge(0)
)

gap_prepared["observed_day"] = gap_prepared[
    "ntl_raw"
].notna()

gap_prepared["temporal_support_pct"] = (
    gap_prepared["observed_day"]
    .rolling(
        ROLLING_DAYS,
        min_periods=1,
    )
    .mean()
    .mul(100)
)

gap_prepared["ntl_30d"] = (
    gap_prepared["ntl_raw"]
    .rolling(
        ROLLING_DAYS,
        min_periods=GAP_MIN_DAYS_30D,
    )
    .mean()
)

gap_prepared["spatial_completeness_pct"] = np.nan
gap_prepared["spatially_qualified_day"] = np.nan
gap_prepared["ntl_30d_strict"] = gap_prepared["ntl_30d"]
gap_prepared["input_type"] = "Gap-filled Black Marble"


In [9]:
# ============================================================
# 5.2 PREPROCESS THE GHSL-MASKED DIRECT OBSERVATIONS
# ============================================================

QUANTILE_COLUMNS = [
    "NTL_min",
    "NTL_p05",
    "NTL_p25",
    "NTL_median",
    "NTL_p75",
    "NTL_p95",
]

QUANTILE_WEIGHTS = np.array(
    [
        0.025,
        0.125,
        0.225,
        0.250,
        0.225,
        0.150,
    ]
)


def approximate_p95_winsorized_mean(frame):
    quantiles = frame[QUANTILE_COLUMNS].to_numpy(
        dtype=float
    )

    valid_rows = np.isfinite(quantiles).all(axis=1)
    ordered_quantiles = np.maximum.accumulate(
        np.where(np.isfinite(quantiles), quantiles, -np.inf),
        axis=1,
    )

    approximated_mean = np.full(
        len(frame),
        np.nan,
        dtype=float,
    )

    approximated_mean[valid_rows] = (
        ordered_quantiles[valid_rows]
        @ QUANTILE_WEIGHTS
    )

    return pd.Series(
        approximated_mean,
        index=frame.index,
    )


rq_prepared = {}

for code_value, mask_label in GHSL_MASKS.items():
    frame = ghsl_raw[code_value].loc[:ANALYSIS_END].copy()

    maximum_valid_pixels = frame["Valid_px"].max()

    frame["ntl_p95_proxy_raw"] = (
        approximate_p95_winsorized_mean(frame)
        .where(frame["Valid_px"].gt(0))
        .where(lambda values: values.ge(0))
    )

    frame["observed_day"] = frame[
        "ntl_p95_proxy_raw"
    ].notna()

    frame["spatial_completeness_pct"] = (
        frame["Valid_px"]
        .div(maximum_valid_pixels)
        .mul(100)
    )

    frame["spatially_qualified_day"] = (
        frame["observed_day"]
        & frame["spatial_completeness_pct"].ge(
            VALID_PIXEL_THRESHOLD_PCT
        )
    )

    frame["temporal_support_pct"] = (
        frame["observed_day"]
        .rolling(
            ROLLING_DAYS,
            min_periods=1,
        )
        .mean()
        .mul(100)
    )

    frame["strict_temporal_support_pct"] = (
        frame["spatially_qualified_day"]
        .rolling(
            ROLLING_DAYS,
            min_periods=1,
        )
        .mean()
        .mul(100)
    )

    frame["ntl_30d"] = (
        frame["ntl_p95_proxy_raw"]
        .rolling(
            ROLLING_DAYS,
            min_periods=RQ_MIN_OBSERVED_DAYS_30D,
        )
        .mean()
    )

    frame["ntl_30d_strict"] = (
        frame["ntl_p95_proxy_raw"]
        .where(frame["spatially_qualified_day"])
        .rolling(
            ROLLING_DAYS,
            min_periods=RQ_MIN_OBSERVED_DAYS_30D,
        )
        .mean()
    )

    frame["input_type"] = "Reliability-qualified DNB-BRDF"
    frame["ghsl_mask"] = mask_label
    rq_prepared[mask_label] = frame


In [10]:
# ============================================================
# 6.1 GAP-FILLED BEFORE/AFTER PREPROCESSING
# ============================================================

fig_gap_preprocessing = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.07,
    row_heights=[0.22, 0.78],
    subplot_titles=(
        "Trailing 30-day data availability",
        "Raw and preprocessed gap-filled nighttime lights",
    ),
)

fig_gap_preprocessing.add_trace(
    go.Heatmap(
        x=gap_prepared.index,
        y=["Gap-filled availability"],
        z=[gap_prepared["temporal_support_pct"].to_numpy()],
        coloraxis="coloraxis",
        zsmooth=False,
        hoverongaps=False,
    ),
    row=1,
    col=1,
)

fig_gap_preprocessing.add_trace(
    go.Scatter(
        x=gap_prepared.index,
        y=gap_prepared["ntl_raw"],
        mode="lines",
        name="Raw gap-filled NTL",
        line=dict(
            color=RQ_RAW_COLOR,
            width=1.0,
        ),
        opacity=0.60,
        connectgaps=False,
    ),
    row=2,
    col=1,
)

fig_gap_preprocessing.add_trace(
    go.Scatter(
        x=gap_prepared.index,
        y=gap_prepared["ntl_30d"],
        mode="lines",
        name=f"30-day mean (minimum {GAP_MIN_DAYS_30D} days)",
        line=dict(
            color=GAP_FILLED_COLOR,
            width=2.4,
        ),
        connectgaps=False,
    ),
    row=2,
    col=1,
)

for row_number in (1, 2):
    fig_gap_preprocessing.add_vline(
        x=EVENT_DATE.to_pydatetime(),
        line=dict(
            color=EVENT_LINE_COLOR,
            width=2,
            dash="dash",
        ),
        row=row_number,
        col=1,
    )

fig_gap_preprocessing.update_xaxes(
    range=[DISPLAY_START, DISPLAY_END],
    title_text="Date",
    row=2,
    col=1,
)

fig_gap_preprocessing.update_yaxes(
    title_text="Mean radiance<br>(nW cm⁻² sr⁻¹)",
    row=2,
    col=1,
)

fig_gap_preprocessing.update_layout(
    template="plotly_white",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="white",
    width=1200,
    height=630,
    title=dict(
        text="Gap-filled preprocessing audit",
        x=0.5,
        xanchor="center",
        font=dict(size=24),
    ),
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.03,
        xanchor="center",
        x=0.5,
        font=dict(size=15),
    ),
    coloraxis=dict(
        colorscale=SC_COLORSCALE,
        cmin=0,
        cmax=100,
        colorbar=dict(
            title="Support (%)",
            thickness=18,
        ),
    ),
    font=dict(
        family="Arial",
        size=16,
        color="#243B5A",
    ),
    margin=dict(l=105, r=90, t=130, b=70),
    hovermode="x unified",
)

fig_gap_preprocessing.show()


In [11]:
# ============================================================
# 6.2 G7 BEFORE/AFTER PREPROCESSING
# ============================================================

g7_prepared = rq_prepared[g7_label]

fig_g7_preprocessing = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.07,
    row_heights=[0.25, 0.75],
    subplot_titles=(
        "Observed-day support and spatial completeness",
        "Raw mean, p95-winsorized proxy, and 30-day series",
    ),
)

fig_g7_preprocessing.add_trace(
    go.Heatmap(
        x=g7_prepared.index,
        y=[
            "Observed-day support",
            "Spatial completeness",
        ],
        z=np.vstack(
            [
                g7_prepared["temporal_support_pct"].to_numpy(),
                g7_prepared["spatial_completeness_pct"].to_numpy(),
            ]
        ),
        coloraxis="coloraxis",
        zsmooth=False,
        hoverongaps=False,
    ),
    row=1,
    col=1,
)

for column, label, color, width, opacity in [
    ("NTL_mean", "Raw daily NTL mean", RQ_RAW_COLOR, 1.0, 0.55),
    (
        "ntl_p95_proxy_raw",
        "Daily p95-winsorized proxy",
        RQ_PROXY_COLOR,
        1.4,
        0.80,
    ),
    (
        "ntl_30d",
        f"30-day proxy (minimum {RQ_MIN_OBSERVED_DAYS_30D} days)",
        RQ_COLOR,
        2.5,
        1.00,
    ),
]:
    fig_g7_preprocessing.add_trace(
        go.Scatter(
            x=g7_prepared.index,
            y=g7_prepared[column],
            mode="lines",
            name=label,
            line=dict(
                color=color,
                width=width,
            ),
            opacity=opacity,
            connectgaps=False,
        ),
        row=2,
        col=1,
    )

for row_number in (1, 2):
    fig_g7_preprocessing.add_vline(
        x=EVENT_DATE.to_pydatetime(),
        line=dict(
            color=EVENT_LINE_COLOR,
            width=2,
            dash="dash",
        ),
        row=row_number,
        col=1,
    )

fig_g7_preprocessing.update_xaxes(
    range=[DISPLAY_START, DISPLAY_END],
    title_text="Date",
    row=2,
    col=1,
)

fig_g7_preprocessing.update_yaxes(
    title_text="Mean radiance<br>(nW cm⁻² sr⁻¹)",
    row=2,
    col=1,
)

fig_g7_preprocessing.update_layout(
    template="plotly_white",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="white",
    width=1200,
    height=650,
    title=dict(
        text=f"Reliability-qualified preprocessing audit: {g7_label}",
        x=0.5,
        xanchor="center",
        font=dict(size=24),
    ),
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.03,
        xanchor="center",
        x=0.5,
        font=dict(size=15),
    ),
    coloraxis=dict(
        colorscale=SC_COLORSCALE,
        cmin=0,
        cmax=100,
        colorbar=dict(
            title="Support (%)",
            thickness=18,
        ),
    ),
    font=dict(
        family="Arial",
        size=16,
        color="#243B5A",
    ),
    margin=dict(l=105, r=90, t=135, b=70),
    hovermode="x unified",
)

fig_g7_preprocessing.show()


In [12]:
# ============================================================
# 6.3 PREPROCESSED GHSL TRAJECTORIES RELATIVE TO BASELINE
# ============================================================

fig_rq_preprocessed = go.Figure()

mask_colors = [
    "#9E9E9E",
    "#7E57C2",
    "#5C6BC0",
    "#29B6F6",
    "#26A69A",
    "#66BB6A",
    RQ_COLOR,
    "#00695C",
]

for (
    mask_label,
    frame,
), color in zip(rq_prepared.items(), mask_colors):
    baseline_median = frame.loc[
        :TRAINING_END,
        "ntl_30d",
    ].median()

    normalized_series = (
        100 * frame["ntl_30d"] / baseline_median
    )

    fig_rq_preprocessed.add_trace(
        go.Scatter(
            x=normalized_series.index,
            y=normalized_series,
            mode="lines",
            name=mask_label,
            line=dict(
                color=color,
                width=(3.0 if mask_label == g7_label else 1.5),
            ),
            opacity=(1.0 if mask_label == g7_label else 0.68),
            connectgaps=False,
        )
    )

fig_rq_preprocessed.add_hline(
    y=100,
    line=dict(
        color=TRAINING_LINE_COLOR,
        width=1.5,
        dash="dot",
    ),
)

fig_rq_preprocessed.add_vline(
    x=EVENT_DATE.to_pydatetime(),
    line=dict(
        color=EVENT_LINE_COLOR,
        width=2,
        dash="dash",
    ),
)

fig_rq_preprocessed.update_xaxes(
    range=[DISPLAY_START, DISPLAY_END],
    title_text="Date",
)

fig_rq_preprocessed.update_yaxes(
    title_text="NTL relative to pre-event training median (%)",
)

fig_rq_preprocessed.update_layout(
    template="plotly_white",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="white",
    width=1200,
    height=600,
    title=dict(
        text="Preprocessed reliability-qualified GHSL trajectories",
        x=0.5,
        xanchor="center",
        font=dict(size=24),
    ),
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.03,
        xanchor="center",
        x=0.5,
        font=dict(size=14),
    ),
    font=dict(
        family="Arial",
        size=16,
        color="#243B5A",
    ),
    margin=dict(l=110, r=60, t=145, b=70),
    hovermode="x unified",
)

fig_rq_preprocessed.show()


**Interpretation.** These figures are the checkpoint between raw inputs and modelling. The gap-filled branch should show whether the support rule prevents isolated null dates from generating long artificial gaps. The GHSL figures should show whether the summary-level p95 proxy removes ordinary mean-versus-median divergence within the Haiyan period. Persistent line breaks or weak spatial completeness remain evidence of limited observability.


## 4. Model-readiness audit

Each training sample requires 90 consecutive preprocessed days: 60 input days and 30 output days. The audit counts eligible sequences twice:

- **relaxed:** directly observed GHSL days with the 18-of-30 temporal rule;
- **strict:** the same series after requiring at least 50% spatial completeness on each contributing day.

The strict count determines whether the paper’s dense-sequence model is evidentially feasible. The relaxed count determines only whether a diagnostic model can be fitted. The three-year history requirement is assessed separately.


In [13]:
# ============================================================
# 7. MODEL-WINDOW CONSTRUCTION AND READINESS AUDIT
# ============================================================

def count_complete_windows(
    series,
    cutoff_date,
):
    values = series.loc[:cutoff_date].to_numpy(
        dtype=float
    )

    required_length = INPUT_WINDOW + OUTPUT_WINDOW

    return sum(
        np.isfinite(
            values[start:start + required_length]
        ).all()
        for start in range(
            max(0, len(values) - required_length + 1)
        )
    )


readiness_rows = []

gap_training_values = gap_prepared.loc[
    :TRAINING_END,
    "ntl_30d",
].dropna()

readiness_rows.append(
    {
        "Series": "Region VIII gap-filled",
        "Input": "Gap-filled Black Marble",
        "Training history (days)": (
            TRAINING_END - gap_prepared.index.min()
        ).days + 1,
        "Relaxed 90-day windows": count_complete_windows(
            gap_prepared["ntl_30d"],
            TRAINING_END,
        ),
        "Strict 90-day windows": count_complete_windows(
            gap_prepared["ntl_30d_strict"],
            TRAINING_END,
        ),
        "Paper three-year minimum met": False,
        "Decision": "Exploratory gap-filled model only",
    }
)

for mask_label, frame in rq_prepared.items():
    relaxed_windows = count_complete_windows(
        frame["ntl_30d"],
        TRAINING_END,
    )

    strict_windows = count_complete_windows(
        frame["ntl_30d_strict"],
        TRAINING_END,
    )

    readiness_rows.append(
        {
            "Series": mask_label,
            "Input": "Reliability-qualified DNB-BRDF",
            "Training history (days)": (
                TRAINING_END - frame.index.min()
            ).days + 1,
            "Relaxed 90-day windows": relaxed_windows,
            "Strict 90-day windows": strict_windows,
            "Paper three-year minimum met": False,
            "Decision": (
                "Strict model feasible"
                if strict_windows > 0
                else "Strict model infeasible; relaxed diagnostic only"
            ),
        }
    )

model_readiness = pd.DataFrame(readiness_rows)
display(model_readiness)


,Series,Input,Training history (days),Relaxed 90-day windows,Strict 90-day windows,Paper three-year minimum met,Decision
0,Region VIII gap-filled,Gap-filled Black Marble,570,457,457,False,Exploratory gap-filled model only
1,G1 (codes 10–30),Reliability-qualified DNB-BRDF,570,188,0,False,Strict model infeasible; relaxed diagnostic only
2,G2 (codes 11–30),Reliability-qualified DNB-BRDF,570,188,0,False,Strict model infeasible; relaxed diagnostic only
3,G3 (codes 12–30),Reliability-qualified DNB-BRDF,570,188,0,False,Strict model infeasible; relaxed diagnostic only
4,G4 (codes 13–30),Reliability-qualified DNB-BRDF,570,188,0,False,Strict model infeasible; relaxed diagnostic only
5,G5 (codes 21–30),Reliability-qualified DNB-BRDF,570,185,0,False,Strict model infeasible; relaxed diagnostic only
6,G6 (codes 22–30),Reliability-qualified DNB-BRDF,570,145,0,False,Strict model infeasible; relaxed diagnostic only
7,G7 (codes 23–30),Reliability-qualified DNB-BRDF,570,131,0,False,Strict model infeasible; relaxed diagnostic only
8,G8 (code 30),Reliability-qualified DNB-BRDF,570,125,0,False,Strict model infeasible; relaxed diagnostic only


**Decision rule.** If strict GHSL windows equal zero, the adaptive model cannot be presented as a reliability-qualified recovery estimator. A relaxed model may still be run to demonstrate how the published algorithm behaves, but its output remains diagnostic and its recovery metrics are suppressed.


## 5. Exploratory adaptive forecasting

The following model retains the paper’s FCNN, 1-D CNN, LSTM, 60-day input, 30-day output, epoch counts, Adam optimizer, mean absolute error loss, median overlapping forecasts, and fixed ensemble weights.

Three safeguards are added transparently:

1. each series is min–max scaled using pre-Haiyan training values only;
2. the anomaly threshold is the 75th percentile of squared ensemble errors in the held-out baseline validation sequences, rather than a threshold contaminated by later years; and
3. forecasts stop one year after Haiyan.

The paper’s ensemble text assigns 0.5 to LSTM and 0.3 to the fully connected model. Its remaining 0.2 label repeats “LSTM”; the remaining weight is interpreted as CNN because the surrounding discussion contrasts stable LSTM and less stable CNN predictions.


In [14]:
# ============================================================
# 8.1 BUILD TRAINING AND FORECAST WINDOWS
# ============================================================

def build_training_windows(series):
    values = series.to_numpy(dtype=float)
    dates = series.index

    x_values = []
    y_values = []
    target_dates = []

    final_start = len(series) - INPUT_WINDOW - OUTPUT_WINDOW + 1

    for start in range(max(0, final_start)):
        input_end = start + INPUT_WINDOW
        output_end = input_end + OUTPUT_WINDOW

        x_window = values[start:input_end]
        y_window = values[input_end:output_end]

        if dates[output_end - 1] > TRAINING_END:
            continue

        if not np.isfinite(x_window).all():
            continue

        if not np.isfinite(y_window).all():
            continue

        x_values.append(x_window)
        y_values.append(y_window)
        target_dates.append(dates[input_end:output_end])

    if not x_values:
        raise ValueError(
            "No complete pre-Haiyan 60-day input and 30-day "
            "output windows are available."
        )

    return (
        np.asarray(x_values, dtype=np.float32),
        np.asarray(y_values, dtype=np.float32),
        target_dates,
    )


def build_forecast_windows(series):
    values = series.to_numpy(dtype=float)
    dates = series.index

    x_values = []
    output_dates = []
    input_ood_pct = []

    final_start = len(series) - INPUT_WINDOW - OUTPUT_WINDOW + 1

    for start in range(max(0, final_start)):
        input_end = start + INPUT_WINDOW
        output_end = input_end + OUTPUT_WINDOW
        x_window = values[start:input_end]

        if not np.isfinite(x_window).all():
            continue

        x_values.append(x_window)
        output_dates.append(dates[input_end:output_end])
        input_ood_pct.append(
            100 * np.mean((x_window < 0) | (x_window > 1))
        )

    if not x_values:
        raise ValueError(
            "No complete 60-day forecast inputs are available."
        )

    return (
        np.asarray(x_values, dtype=np.float32),
        output_dates,
        np.asarray(input_ood_pct, dtype=float),
    )


def aggregate_overlapping_forecasts(
    predicted_windows,
    output_dates,
    values_name="prediction",
):
    rows = []

    for dates, values in zip(output_dates, predicted_windows):
        rows.extend(zip(dates, values))

    return (
        pd.DataFrame(
            rows,
            columns=["date", values_name],
        )
        .groupby("date")[values_name]
        .median()
        .sort_index()
    )


In [15]:
# ============================================================
# 8.2 CHAKRABORTY–STOKES NEURAL-NETWORK ARCHITECTURES
# ============================================================

try:
    import tensorflow as tf

    from tensorflow.keras import Sequential
    from tensorflow.keras.layers import (
        BatchNormalization,
        Conv1D,
        Dense,
        Dropout,
        Flatten,
        Input,
        LSTM,
        MaxPooling1D,
    )
    from tensorflow.keras.optimizers import Adam

except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "This replication requires TensorFlow/Keras. Install TensorFlow "
        "in the notebook kernel before running the modelling sections."
    ) from exc


np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)


def compile_model(model):
    model.compile(
        optimizer=Adam(),
        loss="mae",
    )
    return model


def build_fcnn():
    return compile_model(
        Sequential(
            [
                Input(shape=(INPUT_WINDOW,)),
                Dense(60, activation="relu"),
                Dropout(0.10),
                Dense(45, activation="relu"),
                Dropout(0.10),
                Dense(25, activation="relu"),
                Dropout(0.10),
                Dense(OUTPUT_WINDOW, activation="relu"),
                Dropout(0.10),
            ],
            name="FCNN",
        )
    )


def build_cnn():
    layers = [Input(shape=(INPUT_WINDOW, 1))]

    for filters, kernel_size in [
        (90, 9),
        (45, 9),
        (30, 6),
        (20, 6),
    ]:
        layers.extend(
            [
                Conv1D(
                    filters=filters,
                    kernel_size=kernel_size,
                    padding="same",
                    activation="relu",
                ),
                MaxPooling1D(
                    pool_size=2,
                    padding="same",
                ),
                BatchNormalization(),
                Dropout(0.10),
            ]
        )

    layers.extend(
        [
            Flatten(),
            Dense(20, activation="relu"),
            Dense(15, activation="relu"),
            Dense(OUTPUT_WINDOW, activation="relu"),
        ]
    )

    return compile_model(
        Sequential(layers, name="CNN")
    )


def build_lstm():
    return compile_model(
        Sequential(
            [
                Input(shape=(INPUT_WINDOW, 1)),
                LSTM(
                    45,
                    activation="relu",
                    return_sequences=True,
                ),
                Dropout(0.10),
                LSTM(
                    30,
                    activation="relu",
                ),
                Dropout(0.10),
                Dense(30, activation="relu"),
                Dense(15, activation="relu"),
                Dense(OUTPUT_WINDOW, activation="relu"),
            ],
            name="LSTM",
        )
    )


In [16]:
# ============================================================
# 8.3 TRAIN, VALIDATE, FORECAST, AND FORM THE ENSEMBLE
# ============================================================

def model_input(values, model_name):
    if model_name == "FCNN":
        return values

    return values[..., np.newaxis]


def scale_from_training(series):
    training_values = series.loc[:TRAINING_END].dropna()

    scale_min = training_values.min()
    scale_max = training_values.max()
    scale_range = scale_max - scale_min

    if not np.isfinite(scale_range) or scale_range <= 0:
        raise ValueError(
            "The pre-Haiyan training range is not positive."
        )

    scaled = (series - scale_min) / scale_range

    return scaled, scale_min, scale_max, scale_range


def fit_adaptive_ensemble(
    prepared_frame,
    series_label,
    series_column="ntl_30d",
):
    original_series = prepared_frame.loc[
        :ANALYSIS_END,
        series_column,
    ].copy()

    (
        scaled_series,
        scale_min,
        scale_max,
        scale_range,
    ) = scale_from_training(original_series)

    x_all, y_all, target_dates = build_training_windows(
        scaled_series
    )

    split_index = int(len(x_all) * TRAIN_FRACTION)

    if split_index == 0 or split_index == len(x_all):
        raise ValueError(
            f"Insufficient training/validation split for {series_label}."
        )

    x_train = x_all[:split_index]
    y_train = y_all[:split_index]
    x_validation = x_all[split_index:]
    y_validation = y_all[split_index:]
    validation_dates = target_dates[split_index:]

    tf.keras.backend.clear_session()

    models = {
        "FCNN": build_fcnn(),
        "CNN": build_cnn(),
        "LSTM": build_lstm(),
    }

    training_rows = []
    validation_predictions = {}

    for model_name, model in models.items():
        print(
            f"{series_label}: training {model_name} "
            f"for {MODEL_EPOCHS[model_name]} epochs"
        )

        history = model.fit(
            model_input(x_train, model_name),
            y_train,
            validation_data=(
                model_input(x_validation, model_name),
                y_validation,
            ),
            epochs=MODEL_EPOCHS[model_name],
            batch_size=BATCH_SIZE,
            shuffle=False,
            verbose=0,
        )

        validation_predictions[model_name] = model.predict(
            model_input(x_validation, model_name),
            verbose=0,
        )

        training_rows.append(
            {
                "Series": series_label,
                "Model": model_name,
                "Training windows": len(x_train),
                "Validation windows": len(x_validation),
                "Final training MAE (scaled)": history.history["loss"][-1],
                "Final validation MAE (scaled)": (
                    history.history["val_loss"][-1]
                ),
            }
        )

    validation_ensemble = sum(
        ENSEMBLE_WEIGHTS[model_name]
        * validation_predictions[model_name]
        for model_name in ENSEMBLE_WEIGHTS
    )

    validation_residual = (
        y_validation - validation_ensemble
    ) * scale_range

    validation_mae = np.mean(
        np.abs(validation_residual)
    )

    ensemble_threshold = np.quantile(
        validation_residual.ravel() ** 2,
        1 - ANOMALY_TOP_PERCENT / 100,
    )

    model_thresholds = {}

    for model_name in models:
        model_residual = (
            y_validation
            - validation_predictions[model_name]
        ) * scale_range

        model_thresholds[model_name] = np.quantile(
            model_residual.ravel() ** 2,
            1 - ANOMALY_TOP_PERCENT / 100,
        )

    (
        x_forecast,
        output_dates,
        input_ood_pct,
    ) = build_forecast_windows(scaled_series)

    result = prepared_frame.loc[
        :ANALYSIS_END,
        [
            series_column,
            "observed_day",
            "temporal_support_pct",
            "spatial_completeness_pct",
            "spatially_qualified_day",
        ],
    ].copy()

    result = result.rename(
        columns={series_column: "observed"}
    )

    for model_name, model in models.items():
        predicted_scaled = model.predict(
            model_input(x_forecast, model_name),
            verbose=0,
        )

        predicted_original = (
            predicted_scaled * scale_range + scale_min
        )

        result[f"predicted_{model_name}"] = (
            aggregate_overlapping_forecasts(
                predicted_original,
                output_dates,
            )
        )

    result["predicted_ensemble"] = sum(
        ENSEMBLE_WEIGHTS[model_name]
        * result[f"predicted_{model_name}"]
        for model_name in ENSEMBLE_WEIGHTS
    )

    ood_rows = []

    for dates, ood_pct in zip(output_dates, input_ood_pct):
        ood_rows.extend(
            (date, ood_pct)
            for date in dates
        )

    result["input_ood_pct"] = (
        pd.DataFrame(
            ood_rows,
            columns=["date", "input_ood_pct"],
        )
        .groupby("date")["input_ood_pct"]
        .median()
    )

    result["residual"] = (
        result["observed"]
        - result["predicted_ensemble"]
    )

    result["squared_error"] = result["residual"].pow(2)

    result["ensemble_anomaly"] = (
        result["squared_error"]
        .gt(ensemble_threshold)
        .where(result["squared_error"].notna())
    )

    model_flags = []

    for model_name in models:
        model_squared_error = (
            result["observed"]
            - result[f"predicted_{model_name}"]
        ).pow(2)

        model_flag = model_squared_error.gt(
            model_thresholds[model_name]
        ).where(model_squared_error.notna())

        result[f"anomaly_{model_name}"] = model_flag
        model_flags.append(model_flag.astype(float))

    result["decision_confidence_pct"] = (
        pd.concat(model_flags, axis=1)
        .mean(axis=1, skipna=False)
        .mul(100)
    )

    result["standardized_residual"] = (
        result["residual"] / validation_mae
    )

    plausible_upper = max(
        3 * scale_max,
        scale_max + 5 * scale_range,
    )

    model_stable = (
        result["predicted_ensemble"].dropna().ge(0).all()
        and result["predicted_ensemble"].dropna().le(
            plausible_upper
        ).all()
    )

    result.attrs.update(
        {
            "series_label": series_label,
            "scale_min": scale_min,
            "scale_max": scale_max,
            "validation_mae": validation_mae,
            "ensemble_threshold": ensemble_threshold,
            "model_stable": model_stable,
            "validation_start": validation_dates[0][0],
            "validation_end": validation_dates[-1][-1],
        }
    )

    training_report = pd.DataFrame(training_rows)

    return models, result, training_report


In [17]:
# ============================================================
# 8.4 GUARDED HAIYAN CHANGE AND RECOVERY METRICS
# ============================================================

def first_persistent_recovery(
    recovery_pct,
    threshold_pct,
):
    reached = recovery_pct.ge(threshold_pct).where(
        recovery_pct.notna()
    )

    persistent = (
        reached.astype(float)
        .rolling(
            RECOVERY_PERSISTENCE_DAYS,
            min_periods=RECOVERY_PERSISTENCE_DAYS,
        )
        .sum()
        .eq(RECOVERY_PERSISTENCE_DAYS)
    )

    if not persistent.any():
        return pd.NaT

    persistent_end = persistent[persistent].index[0]

    return (
        persistent_end
        - pd.Timedelta(days=RECOVERY_PERSISTENCE_DAYS - 1)
    )


def summarize_haiyan(
    result,
    series_label,
    input_type,
):
    event_frame = result.loc[
        EVENT_DATE:HAIYAN_METRIC_END
    ].copy()

    detection_end = (
        EVENT_DATE
        + pd.Timedelta(days=DISASTER_DETECTION_DAYS)
    )

    detection_frame = event_frame.loc[
        EVENT_DATE:detection_end
    ]

    negative_anomalies = detection_frame[
        detection_frame["ensemble_anomaly"].eq(True)
        & detection_frame["residual"].lt(0)
    ]

    detected = not negative_anomalies.empty
    model_stable = bool(result.attrs["model_stable"])

    temporal_observation_pct = (
        100 * event_frame["observed_day"].mean()
    )

    median_spatial_completeness = event_frame[
        "spatial_completeness_pct"
    ].median()

    if event_frame["spatially_qualified_day"].notna().any():
        spatially_qualified_days_pct = (
            100
            * event_frame["spatially_qualified_day"]
            .astype(float)
            .mean()
        )
    else:
        spatially_qualified_days_pct = np.nan

    if input_type.startswith("Gap-filled"):
        observability_pass = False
        quality_flag = "Gap-filled diagnostic"
        recovery_permitted = detected and model_stable

    else:
        observability_pass = (
            temporal_observation_pct >= 60
            and pd.notna(median_spatial_completeness)
            and median_spatial_completeness
            >= VALID_PIXEL_THRESHOLD_PCT
        )

        if not model_stable:
            quality_flag = "Model unstable"
        elif not observability_pass:
            quality_flag = "Not observable at declared threshold"
        elif not detected:
            quality_flag = "Observable; no negative anomaly detected"
        else:
            quality_flag = "Reliability-qualified and interpretable"

        recovery_permitted = (
            detected
            and model_stable
            and observability_pass
        )

    diagnostic_frame = detection_frame.dropna(
        subset=["residual"]
    )

    if diagnostic_frame.empty:
        diagnostic_min_date = pd.NaT
        diagnostic_min_residual = np.nan
    else:
        diagnostic_min_date = diagnostic_frame[
            "residual"
        ].idxmin()
        diagnostic_min_residual = diagnostic_frame.loc[
            diagnostic_min_date,
            "residual",
        ]

    first_detection = (
        negative_anomalies.index.min()
        if detected
        else pd.NaT
    )

    impact_date = (
        negative_anomalies["residual"].idxmin()
        if detected
        else pd.NaT
    )

    t50_date = pd.NaT
    t80_date = pd.NaT

    if recovery_permitted:
        impact_severity = -event_frame.loc[
            impact_date,
            "residual",
        ]

        post_impact = event_frame.loc[impact_date:].copy()

        post_impact["recovery_pct"] = (
            1
            - post_impact["residual"]
            .mul(-1)
            .clip(lower=0)
            .div(impact_severity)
        ).mul(100)

        t50_date = first_persistent_recovery(
            post_impact["recovery_pct"],
            50,
        )

        t80_date = first_persistent_recovery(
            post_impact["recovery_pct"],
            80,
        )

    return {
        "Series": series_label,
        "Input": input_type,
        "Quality flag": quality_flag,
        "Model stable": model_stable,
        "Negative disaster anomaly detected": detected,
        "Observed days in Haiyan window (%)": temporal_observation_pct,
        "Median spatial completeness (%)": median_spatial_completeness,
        "Spatially qualified days (%)": spatially_qualified_days_pct,
        "Valid-pixel threshold (%)": VALID_PIXEL_THRESHOLD_PCT,
        "Validation MAE (nW cm⁻² sr⁻¹)": result.attrs[
            "validation_mae"
        ],
        "Diagnostic minimum residual date": diagnostic_min_date,
        "Diagnostic minimum residual": diagnostic_min_residual,
        "First negative anomaly": first_detection,
        "Detection delay (days)": (
            (first_detection - EVENT_DATE).days
            if pd.notna(first_detection)
            else np.nan
        ),
        "Detected impact date": impact_date,
        "Negative anomaly days (0–90)": len(negative_anomalies),
        "Mean model confidence on detected days (%)": (
            negative_anomalies["decision_confidence_pct"].mean()
            if detected
            else np.nan
        ),
        "Recovery metrics permitted": recovery_permitted,
        "T50 date": t50_date,
        "T50 (days from impact)": (
            (t50_date - impact_date).days
            if pd.notna(t50_date)
            else np.nan
        ),
        "T80 date": t80_date,
        "T80 (days from impact)": (
            (t80_date - impact_date).days
            if pd.notna(t80_date)
            else np.nan
        ),
    }


In [18]:
# ============================================================
# 8.5 SHARED FORECAST, SUPPORT, AND ANOMALY PLOT
# ============================================================

def plot_forecast_anomaly(
    result,
    title,
    support_rows,
):
    figure = make_subplots(
        rows=2,
        cols=1,
        shared_xaxes=True,
        vertical_spacing=0.06,
        row_heights=[0.22, 0.78],
        subplot_titles=(
            "Observation support",
            "Observed and expected nighttime lights",
        ),
    )

    figure.add_trace(
        go.Heatmap(
            x=result.index,
            y=[item["label"] for item in support_rows],
            z=np.vstack(
                [
                    result[item["column"]].to_numpy()
                    for item in support_rows
                ]
            ),
            coloraxis="coloraxis",
            zsmooth=False,
            hoverongaps=False,
        ),
        row=1,
        col=1,
    )

    figure.add_trace(
        go.Scatter(
            x=result.index,
            y=result["observed"],
            mode="lines",
            name="Observed 30-day NTL",
            line=dict(
                color=OBSERVED_COLOR,
                width=2.4,
            ),
            connectgaps=False,
        ),
        row=2,
        col=1,
    )

    figure.add_trace(
        go.Scatter(
            x=result.index,
            y=result["predicted_ensemble"],
            mode="lines",
            name="Expected NTL (ensemble)",
            line=dict(
                color=PREDICTED_COLOR,
                width=2.2,
                dash="dash",
            ),
            connectgaps=False,
        ),
        row=2,
        col=1,
    )

    anomaly_points = result[
        result["ensemble_anomaly"].eq(True)
    ]

    figure.add_trace(
        go.Scatter(
            x=anomaly_points.index,
            y=anomaly_points["observed"],
            mode="markers",
            name="Above validation-error threshold",
            marker=dict(
                color=ANOMALY_COLOR,
                size=7,
                symbol="circle-open",
                line=dict(width=1.5),
            ),
            customdata=np.column_stack(
                [
                    anomaly_points["residual"],
                    anomaly_points["decision_confidence_pct"],
                    anomaly_points["input_ood_pct"],
                ]
            ),
            hovertemplate=(
                "%{x|%d %b %Y}<br>"
                "Observed: %{y:.3f}<br>"
                "Residual: %{customdata[0]:.3f}<br>"
                "Model agreement: %{customdata[1]:.0f}%<br>"
                "Out-of-range input: %{customdata[2]:.1f}%"
                "<extra></extra>"
            ),
        ),
        row=2,
        col=1,
    )

    for row_number in (1, 2):
        figure.add_vline(
            x=EVENT_DATE.to_pydatetime(),
            line=dict(
                color=EVENT_LINE_COLOR,
                width=2,
                dash="dash",
            ),
            row=row_number,
            col=1,
        )

        figure.add_vline(
            x=TRAINING_END.to_pydatetime(),
            line=dict(
                color=TRAINING_LINE_COLOR,
                width=1.5,
                dash="dot",
            ),
            row=row_number,
            col=1,
        )

    figure.add_annotation(
        x=(EVENT_DATE + pd.Timedelta(days=5)).to_pydatetime(),
        y=0.05,
        xref="x2",
        yref="y2 domain",
        text="Haiyan",
        showarrow=False,
        xanchor="left",
        font=dict(
            color=EVENT_LINE_COLOR,
            size=16,
        ),
    )

    figure.update_xaxes(
        range=[DISPLAY_START, DISPLAY_END],
        title_text="Date",
        row=2,
        col=1,
    )

    figure.update_yaxes(
        title_text="Mean radiance<br>(nW cm⁻² sr⁻¹)",
        row=2,
        col=1,
    )

    figure.update_layout(
        template="plotly_white",
        paper_bgcolor="rgba(0,0,0,0)",
        plot_bgcolor="white",
        width=1200,
        height=650,
        title=dict(
            text=title,
            x=0.5,
            xanchor="center",
            font=dict(size=24),
        ),
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.03,
            xanchor="center",
            x=0.5,
            font=dict(size=14),
        ),
        coloraxis=dict(
            colorscale=SC_COLORSCALE,
            cmin=0,
            cmax=100,
            colorbar=dict(
                title="Support (%)",
                thickness=18,
            ),
        ),
        font=dict(
            family="Arial",
            size=16,
            color="#243B5A",
        ),
        margin=dict(l=105, r=95, t=135, b=70),
        hovermode="x unified",
    )

    return figure


## 6. Gap-filled exploratory model

This is the closest branch to the published dense gap-filled input. It remains a diagnostic because the supplied regional average is not identical to the paper’s area-weighted functional urban area and the available pre-Haiyan history is shorter than three years.


In [19]:
# ============================================================
# 9.1 FIT AND DISPLAY THE GAP-FILLED MODEL
# ============================================================

(
    gap_models,
    gap_result,
    gap_training_report,
) = fit_adaptive_ensemble(
    prepared_frame=gap_prepared,
    series_label="Region VIII gap-filled",
)

gap_haiyan_summary = pd.DataFrame(
    [
        summarize_haiyan(
            result=gap_result,
            series_label="Region VIII gap-filled",
            input_type="Gap-filled Black Marble",
        )
    ]
)

display(
    gap_training_report.style.format(
        {
            "Final training MAE (scaled)": "{:.4f}",
            "Final validation MAE (scaled)": "{:.4f}",
        }
    )
)

display(gap_haiyan_summary)

fig_gap_forecast = plot_forecast_anomaly(
    result=gap_result,
    title=(
        "Exploratory Chakraborty–Stokes model: "
        "gap-filled Region VIII"
    ),
    support_rows=[
        {
            "label": "Gap-filled availability",
            "column": "temporal_support_pct",
        },
    ],
)

fig_gap_forecast.show()


Region VIII gap-filled: training FCNN for 70 epochs
Region VIII gap-filled: training CNN for 90 epochs
Region VIII gap-filled: training LSTM for 25 epochs


,Series,Model,Training windows,Validation windows,Final training MAE (scaled),Final validation MAE (scaled)
0,Region VIII gap-filled,FCNN,365,92,0.0781,0.0734
1,Region VIII gap-filled,CNN,365,92,0.0384,0.0808
2,Region VIII gap-filled,LSTM,365,92,0.1674,0.1652


,Series,Input,Quality flag,Model stable,Negative disaster anomaly detected,Observed days in Haiyan window (%),Median spatial completeness (%),Spatially qualified days (%),Valid-pixel threshold (%),Validation MAE (nW cm⁻² sr⁻¹),...,First negative anomaly,Detection delay (days),Detected impact date,Negative anomaly days (0–90),Mean model confidence on detected days (%),Recovery metrics permitted,T50 date,T50 (days from impact),T80 date,T80 (days from impact)
0,Region VIII gap-filled,Gap-filled Black Marble,Gap-filled diagnostic,True,True,100.0,NaN,NaN,50.0,0.05326,...,2014-02-02,86,2014-02-03,2,66.666667,True,2014-04-04,60,2014-04-20,76


**Interpretation.** Red markers now represent errors exceeding the threshold learned from held-out pre-Haiyan baseline sequences. The plot is restricted to the event-relevant period. A negative residual after landfall is consistent with an NTL shock, but the 30-day smoother necessarily attenuates and delays an abrupt daily change.


## 7. Relaxed reliability-qualified diagnostic models

The strict readiness audit determines whether these models can support interpretation. When strict windows equal zero, the models below use the relaxed 18-of-30 observed-day series solely to test algorithm behaviour. The observation-quality gate in the summary prevents their numerical outputs from being presented as recovery estimates.


In [20]:
# ============================================================
# 10.1 FIT THE SAME EXPLORATORY MODEL TO ALL GHSL MASKS
# ============================================================

rq_models = {}
rq_results = {}
rq_training_reports = []
rq_haiyan_rows = []

for mask_label, prepared_frame in rq_prepared.items():
    (
        fitted_models,
        fitted_result,
        training_report,
    ) = fit_adaptive_ensemble(
        prepared_frame=prepared_frame,
        series_label=mask_label,
    )

    rq_models[mask_label] = fitted_models
    rq_results[mask_label] = fitted_result
    rq_training_reports.append(training_report)

    rq_haiyan_rows.append(
        summarize_haiyan(
            result=fitted_result,
            series_label=mask_label,
            input_type="Reliability-qualified DNB-BRDF",
        )
    )

rq_training_report = pd.concat(
    rq_training_reports,
    ignore_index=True,
)

rq_haiyan_summary = pd.DataFrame(rq_haiyan_rows)

display(
    rq_training_report.style.format(
        {
            "Final training MAE (scaled)": "{:.4f}",
            "Final validation MAE (scaled)": "{:.4f}",
        }
    )
)


G1 (codes 10–30): training FCNN for 70 epochs
G1 (codes 10–30): training CNN for 90 epochs
G1 (codes 10–30): training LSTM for 25 epochs
G2 (codes 11–30): training FCNN for 70 epochs
G2 (codes 11–30): training CNN for 90 epochs
G2 (codes 11–30): training LSTM for 25 epochs
G3 (codes 12–30): training FCNN for 70 epochs
G3 (codes 12–30): training CNN for 90 epochs
G3 (codes 12–30): training LSTM for 25 epochs
G4 (codes 13–30): training FCNN for 70 epochs
G4 (codes 13–30): training CNN for 90 epochs
G4 (codes 13–30): training LSTM for 25 epochs
G5 (codes 21–30): training FCNN for 70 epochs
G5 (codes 21–30): training CNN for 90 epochs
G5 (codes 21–30): training LSTM for 25 epochs
G6 (codes 22–30): training FCNN for 70 epochs
G6 (codes 22–30): training CNN for 90 epochs
G6 (codes 22–30): training LSTM for 25 epochs
G7 (codes 23–30): training FCNN for 70 epochs
G7 (codes 23–30): training CNN for 90 epochs
G7 (codes 23–30): training LSTM for 25 epochs
G8 (code 30): training FCNN for 70 epochs

,Series,Model,Training windows,Validation windows,Final training MAE (scaled),Final validation MAE (scaled)
0,G1 (codes 10–30),FCNN,150,38,0.1085,0.0868
1,G1 (codes 10–30),CNN,150,38,0.0732,0.1273
2,G1 (codes 10–30),LSTM,150,38,0.1745,0.1672
3,G2 (codes 11–30),FCNN,150,38,0.1235,0.1463
4,G2 (codes 11–30),CNN,150,38,0.0756,0.1768
5,G2 (codes 11–30),LSTM,150,38,0.2329,0.2319
6,G3 (codes 12–30),FCNN,150,38,0.0889,0.0922
7,G3 (codes 12–30),CNN,150,38,0.0562,0.1367
8,G3 (codes 12–30),LSTM,150,38,0.1639,0.1514
9,G4 (codes 13–30),FCNN,150,38,0.1342,0.1142


In [21]:
# ------------------------------------------------------------
# G7 RELAXED DIAGNOSTIC MODEL WITH OBSERVABILITY
# ------------------------------------------------------------

fig_g7_forecast = plot_forecast_anomaly(
    result=rq_results[g7_label],
    title=(
        "Relaxed reliability-qualified diagnostic model: "
        f"{g7_label}"
    ),
    support_rows=[
        {
            "label": "Observed-day support",
            "column": "temporal_support_pct",
        },
        {
            "label": "Spatial completeness",
            "column": "spatial_completeness_pct",
        },
    ],
)

fig_g7_forecast.show()


**Interpretation.** This figure must be read from top to bottom. The model output is numerical; the support panel determines whether it is interpretable. If the spatial-completeness strip remains predominantly below 50%, the correct Haiyan outcome is “not observable at the declared threshold,” regardless of the apparent forecast departure.


## 8. Guarded comparison

The comparison no longer divides residuals by near-zero daily predictions. Residuals are standardized by each model’s held-out baseline validation MAE. A value of $-3$, for example, means observed NTL is three baseline validation errors below the expected trajectory.

Recovery metrics are returned only when all required gates pass:

1. a negative anomaly is detected within 90 days of Haiyan;
2. the model passes the prediction-stability check; and
3. for reliability-qualified inputs, median spatial completeness reaches 50% and at least 60% of days are directly observed.


In [22]:
# ============================================================
# 11.1 GUARDED SUMMARY TABLE
# ============================================================

haiyan_comparison = pd.concat(
    [
        gap_haiyan_summary,
        rq_haiyan_summary,
    ],
    ignore_index=True,
)

comparison_columns = [
    "Series",
    "Input",
    "Quality flag",
    "Model stable",
    "Negative disaster anomaly detected",
    "Observed days in Haiyan window (%)",
    "Median spatial completeness (%)",
    "Spatially qualified days (%)",
    "Validation MAE (nW cm⁻² sr⁻¹)",
    "Diagnostic minimum residual date",
    "Diagnostic minimum residual",
    "First negative anomaly",
    "Detection delay (days)",
    "Negative anomaly days (0–90)",
    "Mean model confidence on detected days (%)",
    "Recovery metrics permitted",
    "T50 (days from impact)",
    "T80 (days from impact)",
]

display(
    haiyan_comparison[comparison_columns]
    .style.format(
        {
            "Observed days in Haiyan window (%)": "{:.1f}",
            "Median spatial completeness (%)": "{:.1f}",
            "Spatially qualified days (%)": "{:.1f}",
            "Validation MAE (nW cm⁻² sr⁻¹)": "{:.3f}",
            "Diagnostic minimum residual": "{:.3f}",
            "Mean model confidence on detected days (%)": "{:.1f}",
        },
        na_rep="—",
    )
)


,Series,Input,Quality flag,Model stable,Negative disaster anomaly detected,Observed days in Haiyan window (%),Median spatial completeness (%),Spatially qualified days (%),Validation MAE (nW cm⁻² sr⁻¹),Diagnostic minimum residual date,Diagnostic minimum residual,First negative anomaly,Detection delay (days),Negative anomaly days (0–90),Mean model confidence on detected days (%),Recovery metrics permitted,T50 (days from impact),T80 (days from impact)
0,Region VIII gap-filled,Gap-filled Black Marble,Gap-filled diagnostic,True,True,100.0,—,—,0.053,2014-02-03 00:00:00,-0.086,2014-02-02 00:00:00,86.000000,2,66.7,True,60.000000,76.000000
1,G1 (codes 10–30),Reliability-qualified DNB-BRDF,Model unstable,False,True,88.4,36.6,45.9,0.027,2013-12-24 00:00:00,-0.062,2013-12-07 00:00:00,29.000000,8,66.7,False,—,—
2,G2 (codes 11–30),Reliability-qualified DNB-BRDF,Model unstable,False,False,88.4,36.7,45.9,0.045,2013-12-24 00:00:00,-0.037,—,—,0,—,False,—,—
3,G3 (codes 12–30),Reliability-qualified DNB-BRDF,Model unstable,False,True,87.3,37.4,44.8,0.041,2013-12-24 00:00:00,-0.098,2013-12-14 00:00:00,36.000000,22,56.1,False,—,—
4,G4 (codes 13–30),Reliability-qualified DNB-BRDF,Model unstable,False,True,87.3,37.6,43.6,0.044,2013-12-24 00:00:00,-0.129,2013-12-16 00:00:00,38.000000,31,87.1,False,—,—
5,G5 (codes 21–30),Reliability-qualified DNB-BRDF,Not observable at declared threshold,True,True,86.2,37.4,43.1,0.055,2013-12-24 00:00:00,-0.156,2013-12-18 00:00:00,40.000000,28,73.8,False,—,—
6,G6 (codes 22–30),Reliability-qualified DNB-BRDF,Not observable at declared threshold,True,True,85.6,33.8,40.9,0.138,2013-12-18 00:00:00,-0.528,2013-12-18 00:00:00,40.000000,30,100.0,False,—,—
7,G7 (codes 23–30),Reliability-qualified DNB-BRDF,Not observable at declared threshold,True,True,85.6,32.0,39.2,0.328,2014-01-14 00:00:00,-0.884,2013-12-23 00:00:00,45.000000,25,66.7,False,—,—
8,G8 (code 30),Reliability-qualified DNB-BRDF,Not observable at declared threshold,True,True,75.1,32.1,38.1,0.962,2014-01-10 00:00:00,-1.632,2014-01-10 00:00:00,63.000000,2,66.7,False,—,—


In [23]:
# ============================================================
# 11.2 BASELINE-ERROR-STANDARDIZED RESIDUAL COMPARISON
# ============================================================

fig_residuals = go.Figure()

fig_residuals.add_trace(
    go.Scatter(
        x=gap_result.index,
        y=gap_result["standardized_residual"],
        mode="lines",
        name="Gap-filled Region VIII",
        line=dict(
            color=GAP_FILLED_COLOR,
            width=2.5,
            dash="dash",
        ),
        connectgaps=False,
    )
)

for (
    mask_label,
    result,
), color in zip(rq_results.items(), mask_colors):
    fig_residuals.add_trace(
        go.Scatter(
            x=result.index,
            y=result["standardized_residual"],
            mode="lines",
            name=mask_label,
            line=dict(
                color=color,
                width=(3.0 if mask_label == g7_label else 1.5),
            ),
            opacity=(1.0 if mask_label == g7_label else 0.65),
            connectgaps=False,
        )
    )

fig_residuals.add_hline(
    y=0,
    line=dict(
        color=TRAINING_LINE_COLOR,
        width=1.5,
        dash="dot",
    ),
)

fig_residuals.add_vline(
    x=EVENT_DATE.to_pydatetime(),
    line=dict(
        color=EVENT_LINE_COLOR,
        width=2,
        dash="dash",
    ),
)

fig_residuals.update_xaxes(
    range=[
        EVENT_DATE - pd.Timedelta(days=180),
        HAIYAN_METRIC_END,
    ],
    title_text="Date",
)

fig_residuals.update_yaxes(
    title_text="Residual / held-out baseline MAE",
)

fig_residuals.update_layout(
    template="plotly_white",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="white",
    width=1200,
    height=610,
    title=dict(
        text="Haiyan departure standardized by baseline model error",
        x=0.5,
        xanchor="center",
        font=dict(size=24),
    ),
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.03,
        xanchor="center",
        x=0.5,
        font=dict(size=14),
    ),
    font=dict(
        family="Arial",
        size=16,
        color="#243B5A",
    ),
    margin=dict(l=110, r=60, t=145, b=70),
    hovermode="x unified",
)

fig_residuals.show()


In [24]:
# ============================================================
# 11.3 NOTEBOOK-READY INTERPRETATION
# ============================================================

gap_row = haiyan_comparison.loc[
    haiyan_comparison["Series"].eq("Region VIII gap-filled")
].iloc[0]

g7_row = haiyan_comparison.loc[
    haiyan_comparison["Series"].eq(g7_label)
].iloc[0]

interpretation_lines = [
    "### 8.1 Preliminary interpretation",
    "",
    (
        "The gap-filled branch is an exploratory algorithmic benchmark "
        f"with quality status **{gap_row['Quality flag']}**."
    ),
    "",
    (
        f"For {g7_label}, directly observed NTL is available on "
        f"**{g7_row['Observed days in Haiyan window (%)']:.1f}%** "
        "of days from landfall through day 180, while median spatial "
        f"completeness is **{g7_row['Median spatial completeness (%)']:.1f}%**."
    ),
    "",
    (
        "The G7 result is therefore classified as "
        f"**{g7_row['Quality flag']}**."
    ),
    "",
]

if g7_row["Recovery metrics permitted"]:
    interpretation_lines.append(
        "The reliability-qualified gates permit recovery metrics."
    )
else:
    interpretation_lines.append(
        "Recovery metrics are suppressed. The correct outcome is "
        "observation-limited, not evidence of non-recovery."
    )

display(Markdown("\n".join(interpretation_lines)))


### 8.1 Preliminary interpretation

The gap-filled branch is an exploratory algorithmic benchmark with quality status **Gap-filled diagnostic**.

For G7 (codes 23–30), directly observed NTL is available on **85.6%** of days from landfall through day 180, while median spatial completeness is **32.0%**.

The G7 result is therefore classified as **Not observable at declared threshold**.

Recovery metrics are suppressed. The correct outcome is observation-limited, not evidence of non-recovery.

## 9. Conclusions and limitations

- The raw-data audit is part of the result. Missing gap-filled dates, extreme GHSL summary values, and weak spatial completeness are shown before modelling.
- The supplied GHSL tables do not permit exact reconstruction of a pixel-level p95-clipped mean. The quantile-based proxy is transparent and should be replaced when pixel-level or upstream clipped exports become available.
- No GHSL mask has a complete pre-Haiyan 90-day sequence after applying the 50% spatial-completeness rule. The reliability-qualified implementation of the published dense-sequence model is therefore not feasible at that threshold.
- Relaxed GHSL models are diagnostic only. Their numerical anomalies cannot override failed observability gates.
- The pre-Haiyan VIIRS history is shorter than the three-year minimum used by Chakraborty and Stokes. All models remain exploratory transfers.
- Training-only scaling prevents raw radiance magnitude from destabilizing the neural networks. Model stability and out-of-range inputs remain visible diagnostics.
- The anomaly threshold is derived from held-out baseline errors and is not contaminated by later urban change or radiance spikes.
- Recovery metrics are suppressed unless a negative disaster anomaly is detected and all applicable quality gates pass.
- Cross-mask comparisons use residuals standardized by held-out baseline MAE, avoiding division by small daily predictions.

**Observability first, interpretation second, recovery metrics third. “Not recovered” and “not observable” remain distinct outcomes.**
